In [1]:
import sys
import pickle
sys.path.append('../../TaskExecutionTimeMining/')
from divide_and_conquer import *
from sklearn.feature_selection import mutual_info_regression

import numpy as np
np.seterr(divide='ignore', invalid='ignore')

{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}

In [2]:
with open("../transformed_event_logs/PCR_start_end_train.pickle", "rb") as f:
    event_log = pickle.load(f)

# numerical attributes : duration, seconds_in_day, day_in_week
numerical_attributes = [
    'duration_seconds',
    'seconds_in_day',
    #'day_of_week',
]

transformed_event_log = event_log.copy()

for num_attr in numerical_attributes:
    transformed_event_log[num_attr] = np.log(transformed_event_log[num_attr]+1)
    transformed_event_log[num_attr] = (transformed_event_log[num_attr] - transformed_event_log[num_attr].mean()) / transformed_event_log[num_attr].std()

/tmp/ipykernel_223923/1997770106.py:2: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  event_log = pickle.load(f)


In [3]:
transformed_event_log

,case:concept:name,id:id_start,cpee:activity_start,cpee:instance_start,lifecycle:transition_start,cpee:lifecycle:transition_start,cpee:state_start,time:timestamp_start,data_start,cpee:description_start,...,seconds_in_day,day_of_week,Callback timeout,Export result,Export to EMS,Match patient data,Receive sample state,Send notification,Wait for plate validation,timeout
20701,10057,a2,a2,1887c54f-6403-420d-87ee-3cbc9ac6888a,start,activity/calling,0.0,2023-04-03 12:03:01.365,"{'value': None, 'children': [('', {'value': ''...",0.0,...,-0.295918,0,0,0,0,0,0,0,0,1
1968,10057,a6,a6,1887c54f-6403-420d-87ee-3cbc9ac6888a,start,activity/calling,0.0,2023-04-03 12:03:01.369,"{'value': None, 'children': [('', {'value': ''...",0.0,...,-0.295918,0,0,0,0,0,0,0,1,1
10073,10057,a4,a4,1887c54f-6403-420d-87ee-3cbc9ac6888a,start,activity/calling,0.0,2023-04-03 12:03:01.374,"{'value': None, 'children': [('', {'value': ''...",0.0,...,-0.295918,0,0,0,0,1,0,0,1,1
16789,10058,a2,a2,1d1323a9-97ea-4ca8-a70f-cc615c04f45f,start,activity/calling,0.0,2023-04-03 12:03:10.071,"{'value': None, 'children': [('', {'value': ''...",0.0,...,-0.295557,0,0,0,0,0,0,0,0,1
25840,10058,a6,a6,1d1323a9-97ea-4ca8-a70f-cc615c04f45f,start,activity/calling,0.0,2023-04-03 12:03:10.079,"{'value': None, 'children': [('', {'value': ''...",0.0,...,-0.295557,0,0,0,0,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2420,18121,a5,a5,e8075204-b208-4666-b262-91a222abd9b2,start,activity/calling,0.0,2023-06-12 21:08:04.345,"{'value': None, 'children': [('', {'value': ''...",0.0,...,0.680531,0,1,1,1,1,1,0,1,1
9978,18101,a5,a5,052a29f7-aa7a-4d86-a164-fc0f182f1cd7,start,activity/calling,0.0,2023-06-12 21:08:04.347,"{'value': None, 'children': [('', {'value': ''...",0.0,...,0.680531,0,1,1,1,1,1,0,1,1
11937,18108,a5,a5,0eae8ae9-f8cf-40fc-a5e3-eb78031df68a,start,activity/calling,0.0,2023-06-12 21:08:04.369,"{'value': None, 'children': [('', {'value': ''...",0.0,...,0.680531,0,1,1,1,1,1,0,1,1
31352,18109,a5,a5,c4c85eaa-560a-4a4e-8df0-38e3fa8a625c,start,activity/calling,0.0,2023-06-12 21:08:04.370,"{'value': None, 'children': [('', {'value': ''...",0.0,...,0.680531,0,1,1,1,1,1,0,1,1


In [4]:
target_column = 'duration_seconds'
continuous_feature_columns = ['seconds_in_day']
nominal_feature_columns = ['concept:name', 'day_of_week']

In [5]:
mi_matrix = calculate_mi_matrix(transformed_event_log, target_column, continuous_feature_columns, nominal_feature_columns,
                                verbose=True)

Computing MI:   0%|          | 0/9 [00:00<?, ?it/s]

MI(day_of_week, day_of_week) = 2.643248924080635
MI(concept:name, concept:name) = 2.6820056820245486
MI(concept:name, day_of_week) = 0.015250796265790977
MI(seconds_in_day, seconds_in_day) = 1.1737605398143152
MI(seconds_in_day, duration_seconds) = 1.1745528430258594
MI(concept:name, seconds_in_day) = 0.7423899561536034
MI(concept:name, duration_seconds) = 1.2224572803974072
MI(day_of_week, duration_seconds) = 0.3208413962239862
MI(day_of_week, seconds_in_day) = 0.6072262444015126


In [6]:
mimr, all_relevance = calculate_maximal_relevance_minimal_redundancy_split(mi_matrix, target_column, continuous_feature_columns, nominal_feature_columns)
print(mimr)
print(all_relevance)

seconds_in_day
{'concept:name': np.float64(0.07590846891609293), 'day_of_week': np.float64(-0.7677339253586599), 'seconds_in_day': np.float64(0.3334272629027156)}
